# Quem Foi? — Experimento de fine-tuning

**Tarefa:** dado um par (pista, depoimento), classificar em `contradiz`, `confirma` ou `neutro`.

**Modelo base:** BERTimbau base (`neuralmind/bert-base-portuguese-cased`).

**Métrica:** F1-macro no conjunto de teste (720 pares de 80 casos que não aparecem no treino).

**Como rodar:** menu *Ambiente de execução → Alterar o tipo de ambiente → GPU T4*, depois *Ambiente de execução → Executar tudo*. Leva uns 5 minutos.

In [ ]:
# 1. Baixa o projeto (com os dados já preparados)
!git clone -q https://github.com/LeviRamoss/Quem-foi-.git projeto || echo 'Se falhar, envie treino.csv e teste.csv pela aba Arquivos'
!pip -q install transformers datasets accelerate scikit-learn

In [ ]:
import pandas as pd, numpy as np, torch, time, json, os
base = 'projeto/fine_tuning/dados/' if os.path.exists('projeto') else ''
treino = pd.read_csv(base + 'treino.csv'); teste = pd.read_csv(base + 'teste.csv')
ROTULOS = ['contradiz', 'confirma', 'neutro']; ID = {r: i for i, r in enumerate(ROTULOS)}
print(len(treino), 'pares de treino |', len(teste), 'pares de teste')
print(treino['rotulo'].value_counts().to_dict())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sem GPU (vai demorar mais)')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
MODELO = 'neuralmind/bert-base-portuguese-cased'
tok = AutoTokenizer.from_pretrained(MODELO)
def preparar(df):
    ds = Dataset.from_pandas(df.assign(label=df['rotulo'].map(ID))[['pista', 'depoimento', 'label']])
    return ds.map(lambda b: tok(b['pista'], b['depoimento'], truncation=True, max_length=160, padding='max_length'), batched=True)
ds_treino, ds_teste = preparar(treino), preparar(teste)

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix
def avaliar(trainer, nome):
    t0 = time.time(); pred = trainer.predict(ds_teste); seg = time.time() - t0
    y_pred = pred.predictions.argmax(-1); y_true = pred.label_ids
    f1 = f1_score(y_true, y_pred, average='macro')
    print(f'== {nome}: F1-macro = {f1:.3f}  ({seg/len(teste)*1000:.1f} ms por par)')
    print(classification_report(y_true, y_pred, target_names=ROTULOS, digits=3))
    print(confusion_matrix(y_true, y_pred))
    return round(float(f1), 3), round(seg / len(teste) * 1000, 2)

import inspect
# 2 épocas de 180 passos (2.880 pares / lote 16); aquecimento em 10% = 36 passos.
# Só passa as opções que a versão instalada do transformers aceita.
cfg = dict(output_dir='saida', num_train_epochs=2, per_device_train_batch_size=16, per_device_eval_batch_size=64,
           learning_rate=2e-5, weight_decay=0.01, warmup_steps=36, logging_steps=50, save_strategy='no',
           report_to='none', fp16=torch.cuda.is_available(), seed=42)
aceitas = inspect.signature(TrainingArguments.__init__).parameters
args = TrainingArguments(**{k: v for k, v in cfg.items() if k in aceitas})

# 2. Modelo BASE, sem fine-tuning (a camada de classificação ainda não aprendeu nada)
torch.manual_seed(42)
modelo = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=3)
trainer = Trainer(model=modelo, args=args, train_dataset=ds_treino)
f1_base, ms_base = avaliar(trainer, 'BERTimbau BASE (sem fine-tuning)')

In [ ]:
# 3. Fine-tuning
t0 = time.time(); trainer.train(); minutos = (time.time() - t0) / 60
print(f'Treino: {minutos:.1f} min')
f1_ft, ms_ft = avaliar(trainer, 'BERTimbau com FINE-TUNING')

In [ ]:
# 4. Resumo para o relatório
res = {'modelo': MODELO, 'pares_treino': len(treino), 'pares_teste': len(teste), 'epocas': 2,
       'f1_base_sem_treino': f1_base, 'f1_fine_tuning': f1_ft, 'minutos_de_treino': round(minutos, 1),
       'ms_por_par_fine_tuning': ms_ft, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}
print(json.dumps(res, indent=1, ensure_ascii=False))
json.dump(res, open('resultados_bertimbau.json', 'w'), indent=1, ensure_ascii=False)
print('\nTire um print desta célula e da anterior para o relatório.')